# Mock lab demo, A3: model-driven batches

This notebook walks through one A3 batch with the two roles of the protocol:

| Step | Who | How |
| --- | --- | --- |
| 1. Generate the batch | Data scientist | `A3/A03.py` rebuilds Ax from the spreadsheet and adds 4 welds (plus a control weld in A3 batches 2 and 5). |
| 2. Weld, rate quality, read stress | Lab | The widget below; type the values into the yellow columns of `data/experiments.xlsx`. |
| 3. Decide which welds to test | Data scientist | `A3/A03.py` scores the welds and writes D3 (test), D4 (leave uncut) or D2 (audit) into the purple columns. |
| 4. Tensile-test | Lab | The widget: only D3, D2 and control welds. |
| 5. Next batch | Data scientist | `A3/A03.py` again. With `--early` it can generate before step 4 is finished. |

The scores are a **placeholder** (seeded random numbers) until the EHVI calculation exists,
so the decisions show how the rules work, not which welds are really promising.

Excel must be **saved and closed** before each data scientist step, because the script writes to the file.

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "_mock").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from common import msvc, settings
from _mock.lab import MockLab
from _mock.lab_widget import LabDemo

# The scripts below inherit this environment, so Ax gets the fast BoTorch kernels.
print("MSVC compiler available:", msvc.ensure(verbose=True))
lab = MockLab()


def run(script, *args):
    """Run a phase script as the data scientist would, and show its output."""
    result = subprocess.run(
        [sys.executable, str(ROOT / script), *args], capture_output=True, text=True, cwd=ROOT
    )
    print(result.stdout, result.stderr)

## Before A3: A1 must be complete

A3 starts only when every A1 weld is complete. To demo A3 without doing A1 by hand,
set `PLAY_A1 = True`: it creates the A1 design (if there is none) and plays the lab for all of it.

In [ ]:
PLAY_A1 = False

if PLAY_A1:
    run("A1/A01.py")
    run("A1/A01_mock_fill.py", "--all")
    run("A1/A01.py")

## Step 1. Data scientist: generate the batch

Takes about half a minute (Ax fits its model). The new welds are added to the spreadsheet.

In [ ]:
run("A3/A03.py")

## Steps 2 and 4. Lab: weld and measure

Pick the new welds (marked A3). For each: **Weld**, **Visual inspection**, **Read stress** at every position.
The panel under the settings says what to do with the specimens:

* *not decided yet*: stop after stress, type the values into the sheet, and wait for step 3;
* *D3* or *D2*: cut and tensile-test both specimens;
* *D4*: leave the weld uncut (the lab refuses to cut unless you tick the override).

Press **Reload sheet** after each data scientist step to see the new welds and decisions.

In [ ]:
LabDemo(settings.EXPERIMENT_FILE, lab).show()

## Step 3. Data scientist: decide which welds to test

Run this once the stress of every weld in the batch is in the sheet. It writes the
decisions and the tensile quota balance. Then press **Reload sheet** in the widget and do step 4.

In [ ]:
run("A3/A03.py")

## Step 5. Data scientist: next batch

Once the tensile results are in, `A03.py` checks them and adds the next batch.

To keep the lab busy while the tensile tests are still running, use `--early`: the next batch is generated
as soon as the current one has its test decisions. Its welds still waiting for UTS go to Ax as pending
trials with their stress. At most `settings.A3.MAX_BATCHES_WAITING` batches may wait like that.

In [ ]:
EARLY = False

run("A3/A03.py", *(["--early"] if EARLY else []))

## Shortcuts

* Let the mock lab do every lab step that is allowed right now, instead of clicking:
  `run("A3/A03_mock_fill.py")`.
* Play the whole of A3 automatically (a few minutes): `run("A3/A03_mock_fill.py", "--run-all")`.
* Start over: in a terminal, `python _mock/reset_demo.py` (asks for confirmation), then restart this kernel.

In [ ]:
run("A3/A03_mock_fill.py")